# EDA — Baches de Hermosillo

Análisis exploratorio del dataset final de reportes de baches.

## Objetivos
- Entender estructura y calidad básica.
- Analizar evolución temporal.
- Identificar colonias con más reportes.
- Revisar distribución espacial.
- Analizar origen de colonia.
- Explorar material y texto.
- Detectar posibles reportes repetidos.

> Importante: el dataset representa reportes ciudadanos, no necesariamente baches únicos.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)


## 1. Cargar dataset


In [ ]:
DATA_PATH = Path("../data/processed/bachometro/baches_hermosillo_colonia_filled.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())


In [ ]:
df.info()


## 2. Perfil general


In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_pct": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(dropna=True),
})
display(quality.sort_values(["nulos_pct", "unicos"], ascending=[False, False]))


In [ ]:
print("Filas duplicadas:", df.duplicated().sum())
print("IDs duplicados:", df["id"].duplicated().sum())
print("¿id único?:", df["id"].is_unique)


## 3. Preparación mínima


In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.day_name()
df["description_length"] = df["description"].fillna("").str.len()
df["description_words"] = df["description"].fillna("").str.split().str.len()
print("Fechas inválidas:", df["date"].isna().sum())


## 4. Análisis temporal


In [ ]:
reports_by_year = df.groupby("year").size().rename("reportes")
display(reports_by_year.to_frame())


In [ ]:
plt.figure(figsize=(9,5))
reports_by_year.plot(kind="bar")
plt.title("Reportes por año")
plt.xlabel("Año")
plt.ylabel("Reportes")
plt.xticks(rotation=0)
plt.show()


In [ ]:
reports_by_month = df.groupby("month").size().rename("reportes")
display(reports_by_month.to_frame())


In [ ]:
plt.figure(figsize=(9,5))
reports_by_month.plot(kind="bar")
plt.title("Reportes por mes")
plt.xlabel("Mes")
plt.ylabel("Reportes")
plt.xticks(rotation=0)
plt.show()


In [ ]:
monthly_series = df.set_index("date").resample("ME").size()
plt.figure(figsize=(14,5))
monthly_series.plot()
plt.title("Evolución mensual de reportes")
plt.xlabel("Fecha")
plt.ylabel("Reportes")
plt.show()


## 5. Colonias


In [ ]:
top_colonias = df["colonia"].value_counts().head(20)
display(top_colonias.to_frame("reportes"))


In [ ]:
plt.figure(figsize=(10,7))
top_colonias.sort_values().plot(kind="barh")
plt.title("Top 20 colonias por reportes")
plt.xlabel("Reportes")
plt.ylabel("Colonia")
plt.show()


In [ ]:
colonia_year = df.groupby(["year","colonia"]).size().reset_index(name="reportes")
display(colonia_year.sort_values("reportes", ascending=False).head(50))


## 6. Origen de colonia


In [ ]:
display(df["colonia_source"].value_counts(dropna=False).to_frame("reportes"))
display((df["colonia_source"].value_counts(normalize=True, dropna=False)*100).round(2).to_frame("porcentaje"))


In [ ]:
source_by_year = pd.crosstab(df["year"], df["colonia_source"], normalize="index").mul(100).round(2)
display(source_by_year)


## 7. Material


In [ ]:
display(df["material"].value_counts(dropna=False).to_frame("reportes"))
display(pd.crosstab(df["year"], df["material"]))


## 8. Distribución espacial


In [ ]:
plt.figure(figsize=(9,9))
plt.scatter(df["longitude"], df["latitude"], s=5, alpha=0.15)
plt.title("Distribución espacial de reportes")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.show()


In [ ]:
plt.figure(figsize=(9,9))
hb = plt.hexbin(df["longitude"], df["latitude"], gridsize=70, mincnt=1)
plt.title("Densidad espacial de reportes")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.colorbar(hb, label="Número de reportes")
plt.show()


## 9. Texto de las descripciones


In [ ]:
display(df[["description_length","description_words"]].describe(percentiles=[.25,.5,.75,.9,.95,.99]))


In [ ]:
plt.figure(figsize=(10,5))
df["description_words"].clip(upper=df["description_words"].quantile(.99)).hist(bins=50)
plt.title("Número de palabras por reporte")
plt.xlabel("Palabras")
plt.ylabel("Reportes")
plt.show()


## 10. Posibles reportes repetidos


In [ ]:
coord_counts = df.groupby(["latitude","longitude"]).size().sort_values(ascending=False)
display(coord_counts.head(30).to_frame("reportes"))
print("Coordenadas exactas repetidas:", (coord_counts > 1).sum())


In [ ]:
same_point_date = df.groupby(["latitude","longitude","date"]).size().sort_values(ascending=False)
display(same_point_date[same_point_date > 1].head(30).to_frame("reportes"))


## 11. Resumen automático


In [ ]:
print("Total reportes:", len(df))
print("Periodo:", df["date"].min(), "a", df["date"].max())
print("Colonias distintas:", df["colonia"].nunique())
print("\nTop 5 colonias:")
print(df["colonia"].value_counts().head(5))
print("\nFuente de colonia:")
print(df["colonia_source"].value_counts())
print("\nAño con más reportes:", df["year"].value_counts().idxmax())
print("Coordenadas repetidas:", (coord_counts > 1).sum())


## 12. Preguntas para interpretar

1. ¿En qué años aumentan o disminuyen los reportes?
2. ¿Existe estacionalidad mensual?
3. ¿Qué colonias concentran más reportes?
4. ¿Las mismas colonias aparecen consistentemente entre años?
5. ¿Hay clusters espaciales claros?
6. ¿Qué porcentaje de colonias fue original vs inferido?
7. ¿Existen ubicaciones repetidas que sugieran múltiples reportes del mismo incidente?
8. ¿Hay patrones por material?
9. ¿Las descripciones tienen suficiente señal para NLP?
10. ¿Qué variables pueden convertirse en features?
